### First Training Attempt Without Using Strucural Data From The Graph

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import linear, relu, sigmoid
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

/Users/arnemallon/Developer/BlockChainAnalysis/backend/notebook_venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
data = pd.read_csv("/Users/arnemallon/Developer/Datasets/BABD-13.csv")

print("Shape of the dataset:", data.shape)
print("\nFirst 5 rows of the dataset:")
print(data.head())

Shape of the dataset: (544462, 151)

First 5 rows of the dataset:
                              account  SW   PAIa11-1   PAIa11-2    PAIa12  \
0  13AM4VW2dhxYgXeQepoHkHSQuy6NgaEb94  SA   0.024182   0.000000 -0.024182   
1  12t9YDPgwueZ9NyMgw519p7AA8isjr6SMw  SA   0.217012   0.000000 -0.217012   
2  115p7UMMngoj1pMvkpHijcRdfJNXj6LrLn  SA   0.067596   0.000000 -0.067596   
3  1NXTgfGprVktuokv3ZLhGCPCjcKjXbswAM  SA   0.000095   0.000000 -0.000095   
4  15RLWdVnY5n1n7mTvU1zjg67wt86dhYqNj  SA  25.000993  25.000993  0.000000   

   PAIa13   PAIa14-1   PAIa14-2   PAIa14-3   PAIa14-4  ...  S2-2  S2-3  \
0     0.0   0.001000   0.021682   0.000000   0.000000  ...  1469  2725   
1     0.0   0.000005   0.037723   0.000000   0.000000  ...     1    95   
2     0.0   0.000278   0.037200   0.000000   0.000000  ...   765  1751   
3     0.0   0.000041   0.000054   0.000000   0.000000  ...     1     2   
4     1.0  25.000993  25.000993  25.000993  25.000993  ...     5    11   

         S3       S4      

In [3]:
# we only need the top 8 features selected in feature_selection

# Select specific columns from the dataset
#X = data[['PAIa13', 'S2-3', 'CI2a32-2', 'CI2a32-4', 'S2-2', 'PAIa11-1', 'PAIa11-2', 'S2-1']]
X = data[['S2-1', 'PTIa41-2', 'PTIa41-3', 'S4', 'CI2a32-2', 'PTIa21', 'CI3a12-3', 'PAIa13']]
y = data['label']

X = X.to_numpy()
y = y.to_numpy()

# Display the shape and first 5 rows of the selected data
print("Shapes of the selected data:", X.shape, y.shape)


Shapes of the selected data: (544462, 8) (544462,)


In [4]:
from sklearn.model_selection import train_test_split

# Get 60% of the dataset as the training set. Put the remaining 40% in temporary variables.
x_train, x_, y_train, y_ = train_test_split(X, y, test_size=0.20, random_state=1)

# Split the 40% subset above into two: one half for cross validation and the other for the test set
x_cv, x_test, y_cv, y_test = train_test_split(x_, y_, test_size=0.9999, random_state=1)

# Delete temporary variables
del x_, y_


print(f"the shape of the training set (input) is: {x_train.shape}")
print(f"the shape of the training set (target) is: {y_train.shape}\n")
print(f"the shape of the cross validation set (input) is: {x_cv.shape}")
print(f"the shape of the cross validation set (target) is: {y_cv.shape}\n")
print(f"the shape of the test set (input) is: {x_test.shape}")
print(f"the shape of the test set (target) is: {y_test.shape}")

the shape of the training set (input) is: (435569, 8)
the shape of the training set (target) is: (435569,)

the shape of the cross validation set (input) is: (10, 8)
the shape of the cross validation set (target) is: (10,)

the shape of the test set (input) is: (108883, 8)
the shape of the test set (target) is: (108883,)


In [5]:
# Scale the features

# Initialize the class
scaler_linear = StandardScaler()

# Compute the mean and standard deviation of the training set then transform it
x_train_scaled = scaler_linear.fit_transform(x_train)
x_cv_scaled = scaler_linear.transform(x_cv)
x_test_scaled = scaler_linear.transform(x_test)

In [6]:
tf.random.set_seed(1234)

model1 = tf.keras.Sequential([
    tf.keras.Input(shape=(8,)),

    Dense(units=36, activation = 'relu', name = 'layer1'),
    Dense(units=24, activation = 'relu', name = 'layer2'),
    Dense(units=16, activation = 'relu', name = 'layer3'),
    Dense(units=13, activation = 'linear', name = 'output'),
], name = 'non_structural_model_1')

model1.summary()

Model: "non_structural_model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 layer1 (Dense)              (None, 36)                324       
                                                                 
 layer2 (Dense)              (None, 24)                888       
                                                                 
 layer3 (Dense)              (None, 16)                400       
                                                                 
 output (Dense)              (None, 13)                221       
                                                                 
Total params: 1833 (7.16 KB)
Trainable params: 1833 (7.16 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [7]:
tf.random.set_seed(1234)

model2 = tf.keras.Sequential([
    tf.keras.Input(shape=(8,)),

    Dense(units=12, activation = 'relu', name = 'layer0'),
    Dense(units=16, activation = 'relu', name = 'layer1'),
    Dense(units=24, activation = 'relu', name = 'layer2'),
    Dense(units=36, activation = 'relu', name = 'layer3'),
    Dense(units=48, activation = 'relu', name = 'layer4'),
    Dense(units=60, activation = 'relu', name = 'layer5'),
    Dense(units=72, activation = 'relu', name = 'layer6'),
    Dense(units=60, activation = 'relu', name = 'layer7'),
    Dense(units=48, activation = 'relu', name = 'layer8'),
    Dense(units=36, activation = 'relu', name = 'layer9'),
    Dense(units=24, activation = 'relu', name = 'layer10'),
    Dense(units=16, activation = 'relu', name = 'layer11'),
    Dense(units=13, activation = 'linear', name = 'output'),
], name = 'non_structural_model_2')

model2.summary()

Model: "non_structural_model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 layer0 (Dense)              (None, 12)                108       
                                                                 
 layer1 (Dense)              (None, 16)                208       
                                                                 
 layer2 (Dense)              (None, 24)                408       
                                                                 
 layer3 (Dense)              (None, 36)                900       
                                                                 
 layer4 (Dense)              (None, 48)                1776      
                                                                 
 layer5 (Dense)              (None, 60)                2940      
                                                                 
 layer6 (Dense)              (None, 72)     

In [8]:
tf.random.set_seed(1234)

model3 = tf.keras.Sequential([
    tf.keras.Input(shape=(8,)),

    Dense(units=24, activation = 'relu', name = 'layer1'),
    Dense(units=32, activation = 'relu', name = 'layer2'),
    Dense(units=28, activation = 'relu', name = 'layer3'),
    Dense(units=24, activation = 'relu', name = 'layer4'),
    Dense(units=20, activation = 'relu', name = 'layer5'),
    Dense(units=16, activation = 'relu', name = 'layer6'),
    Dense(units=13, activation = 'linear', name = 'output'),
], name = 'non_structural_model_3')

model3.summary()

Model: "non_structural_model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 layer1 (Dense)              (None, 24)                216       
                                                                 
 layer2 (Dense)              (None, 32)                800       
                                                                 
 layer3 (Dense)              (None, 28)                924       
                                                                 
 layer4 (Dense)              (None, 24)                696       
                                                                 
 layer5 (Dense)              (None, 20)                500       
                                                                 
 layer6 (Dense)              (None, 16)                336       
                                                                 
 output (Dense)              (None, 13)     

In [9]:
train_errors = []
cv_errors = []

#for model in [model1, model2, model3]:
for model in [model2]:
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=0.001), #0.001
    )

    print(f"Model {model.name} has been compiled")

    history = model.fit(
        x_train_scaled,y_train,
        epochs=200,
        verbose=1
    )

    print(f"Model {model.name} has been trained\n")

    # Calculate train error
    yhat = model.predict(x_train_scaled)
    yhat = tf.nn.softmax(yhat)
    yhat = np.argmax(yhat, axis=1)
    train_error = np.mean(y_train != yhat)
    train_errors.append(train_error)

    # Calculate cv error
    yhat = model.predict(x_cv_scaled)
    yhat = tf.nn.softmax(yhat)
    yhat = np.argmax(yhat, axis=1)
    cv_error = np.mean(y_cv != yhat)
    cv_errors.append(cv_error)

    print('\n\n')

for model_num in range(len(train_errors)):
    print(f"Model {model_num+1}: train error: {train_errors[model_num]:.5f}, cv error: {cv_errors[model_num]:.5f}")
    


Model non_structural_model_2 has been compiled
Epoch 1/200
13612/13612 [==============================] - 21s 1ms/step - loss: 0.9038
Epoch 2/200
13612/13612 [==============================] - 18s 1ms/step - loss: 0.8105
Epoch 3/200
13612/13612 [==============================] - 17s 1ms/step - loss: 0.7776
Epoch 4/200
13612/13612 [==============================] - 18s 1ms/step - loss: 0.7551
Epoch 5/200
13612/13612 [==============================] - 19s 1ms/step - loss: 0.7356
Epoch 6/200
13612/13612 [==============================] - 18s 1ms/step - loss: 0.7217
Epoch 7/200
13612/13612 [==============================] - 17s 1ms/step - loss: 0.7112
Epoch 8/200
13612/13612 [==============================] - 16s 1ms/step - loss: 0.6988
Epoch 9/200
13612/13612 [==============================] - 16s 1ms/step - loss: 0.6907
Epoch 10/200
13612/13612 [==============================] - 18s 1ms/step - loss: 0.6856
Epoch 11/200
13612/13612 [==============================] - 17s 1ms/step - loss: 0

In [10]:
best_model = np.argmin(cv_errors) + 1

yhat = model.predict(x_test_scaled)
yhat = tf.nn.softmax(yhat)
yhat = np.argmax(yhat, axis=1)
test_error = np.mean(y_test != yhat)

print(f"\nThe best model is model {best_model}\n")
print(f"training error: {train_errors[best_model-1]:.5f}")
print(f"cv error: {cv_errors[best_model-1]:.5f}")
print(f"test error: {test_error:.5f}")

3403/3403 [==============================] - 2s 467us/step

The best model is model 1

training error: 0.19252
cv error: 0.10000
test error: 0.19178


#### Bisheriger Rekord

- test error: 0.15045

-> sieht noch nach high bias aus, da die errors ziemlich nah aneinander liegen, aber noch nicht sonderlich gut sind

Dense(units=40, activation = 'relu', name = 'layer1'),
    Dense(units=80, activation = 'relu', name = 'layer2'),
    Dense(units=120, activation = 'relu', name = 'layer3'),
    Dense(units=96, activation = 'relu', name = 'layer4'),
    Dense(units=80, activation = 'relu', name = 'layer5'),
    Dense(units=64, activation = 'relu', name = 'layer6'),
    Dense(units=48, activation = 'relu', name = 'layer7'),
    Dense(units=32, activation = 'relu', name = 'layer8'),
    Dense(units=13, activation = 'linear', name = 'output'),

    0,0005


In [11]:
# New cell content
import json

# The scaler was named scaler_linear in your notebook
# Make sure to run the cell where it was created and fitted first!
scaler_data = {
    'mean': scaler_linear.mean_.tolist(),
    'scale': scaler_linear.scale_.tolist()
}

with open('../../ml-models/scaler.json', 'w') as f:
    json.dump(scaler_data, f)

print("✅ Scaler data saved successfully to ml-models/scaler.json")

✅ Scaler data saved successfully to ml-models/scaler.json


In [12]:
# The best model was named model2 in your notebook
# Make sure it has been trained by running the training cell first!
model2.save('../../ml-models/bitcoin_classifier.keras')

print("✅ Model saved successfully to ml-models/bitcoin_classifier.keras")

✅ Model saved successfully to ml-models/bitcoin_classifier.keras
